# Análise exploratória de dengue (2014–2021)

A análise percorre uma partição por vez e mantém apenas agregados em memória. Isso evita concatenar quase 10 milhões de casos.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path('../..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from dengue_pipeline.paths import DENGUE_YEARS, analysis_dataset_path

In [ ]:
annual = []
monthly = []
for year in DENGUE_YEARS:
    frame = pd.read_parquet(
        analysis_dataset_path(year),
        columns=['notification_month', 'final_classification'],
    )
    annual.append({
        'year': year,
        'cases': len(frame),
        'confirmed': int(frame['final_classification'].sum()),
        'discarded': int((frame['final_classification'] == 0).sum()),
    })
    grouped = frame.groupby(['notification_month', 'final_classification'], dropna=False).size().rename('cases').reset_index()
    grouped.insert(0, 'year', year)
    monthly.append(grouped)

annual_summary = pd.DataFrame(annual)
monthly_summary = pd.concat(monthly, ignore_index=True)
annual_summary

In [ ]:
ax = annual_summary.set_index('year')[['confirmed', 'discarded']].plot.bar(figsize=(12, 5), stacked=True)
ax.set(title='Casos rotulados de dengue por ano — 2014 a 2021', xlabel='Ano', ylabel='Casos')